# Semantic Diffusion: budgeted LLM edge weights for community detection

CBCA315 Minor Project, Topic 2. This notebook runs the same code as `make all`, step by step, and shows the results.

**Before you start (Colab):**
1. Put the project folder in Google Drive at `MyDrive/CBCA315-Semantic-Diffusion`.
2. Add your OpenRouter key: click the key icon (Secrets) on the left, name it `OPENROUTER_API_KEY`, turn on notebook access.

The LLM answers are cached in `outputs/llm/calls.jsonl`, so re-running costs nothing.

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/CBCA315-Semantic-Diffusion")
    !pip -q install -r requirements.txt
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
else:
    os.chdir(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

sys.path.insert(0, "src")
import config, load_data, edge_weights, llm_scoring, detect, diffusion, make_charts
import pandas as pd
from IPython.display import Image, Markdown, display
print("Project folder:", Path.cwd())

## Step 1. Data: Cora and PubMed with their original titles and abstracts
Text is joined to each node **by paper ID** (not by row position).

In [ ]:
load_data.run()
g = load_data.load("cora")
print(g.texts[0][:400])

## Step 2. Edge weights W0 (all 1) and W1 (text cosine)
PubMed embeddings take about 10-15 minutes on a CPU the first time, then they are cached.

In [ ]:
edge_weights.run()
display(Image("outputs/charts/fig3_cosine_band.png")) if Path("outputs/charts/fig3_cosine_band.png").exists() else None

## Step 3. LLM re-scores the ambiguous edges (W2)
Only edges with cosine in [0.3, 0.6], at most 10% of edges. Cached answers are reused; spending stops at $1.

In [ ]:
llm_scoring.run()

## Step 4. Community detection: 3 detectors × 5 seeds × every weight variant

In [ ]:
detect.run()
runs = pd.read_csv("outputs/runs.csv")
runs[runs.variant.isin(["W0", "W1", "W2"])].groupby(["dataset", "detector", "variant"])[["NMI", "ARI", "ACC", "F1", "Q"]].mean().round(3)

## Step 5. Downstream task: influence spread (Independent Cascade)

In [ ]:
diffusion.run()

## Step 6. Tables, significance tests and charts

In [ ]:
make_charts.run()
for t in ["t2_main_results", "t3_significance", "t5_budget_vs_nmi", "t8_diffusion"]:
    display(Markdown(f"### {t}\n" + Path(f"outputs/tables/{t}.md").read_text()))
for f in sorted(Path("outputs/charts").glob("*.png")):
    display(Image(str(f), width=800))